# Correlation Mapping:

### Correlation Matrix:

In [1]:
import pandas as pd
import numpy as np

# Constants from metrics doc
MIN_TICKERS = 5

FILE_PATH = "../../data/processed/gdelt_ohlcv_join.parquet"

df = pd.read_parquet(FILE_PATH)

# Check data sparsity 
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
print("Date range:", df["price_date"].min(), "->", df["price_date"].max())
print("Unique ticker-days:", df[["ticker", "price_date"]].drop_duplicates().shape[0])

# Parse dates
df["price_date"] = pd.to_datetime(df["price_date"], utc=True)
df["article_date"] = pd.to_datetime(df["article_date"], utc=True)

# Ensure numeric
num_cols = [
    "sentiment_score",
    "next_open",
    "next_close"
]

for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Returns using metrics doc formula
# r_i,t = (close - open) / open

df["ret"] = (df["next_close"] - df["next_open"]) / df["next_open"]


# Aggregate sentiment to daily level (t)

daily_sentiment = (
    df.groupby("price_date")
      .agg(
          sentiment_score=("sentiment_score", "mean")
      )
      .reset_index()
)

# Compute cross-sectional metrics per day

returns = (
    df.groupby(["ticker", "price_date"])
      .agg(ret=("ret", "first"))
      .reset_index()
)

cs_stats = (
    returns.groupby("price_date")
    .agg(
        ret_cs_std=("ret", "std"),
        ret_mean=("ret", "mean"),
        n_tickers_returns=("ticker", "nunique"),
        ret_cs_mad=("ret", lambda x: np.mean(np.abs(x - np.mean(x))))
    )
    .reset_index()
)

# coverage ratio (7 tickers total)
cs_stats["coverage_ratio"] = cs_stats["n_tickers_returns"] / 7

# ticker coverage threshold
cs_stats = cs_stats[cs_stats["n_tickers_returns"] >= MIN_TICKERS]


# Merge sentiment with dispersion metrics

corr_df = daily_sentiment.merge(cs_stats, on="price_date")

corr_df = corr_df.dropna()


# Correlation matrices

cols = [
    "sentiment_score",
    "ret_cs_std",
    "ret_cs_mad",
    "ret_mean"
]

print("\nPearson Correlation Matrix\n")
print(corr_df[cols].corr(method="pearson"))

print("\nSpearman Correlation Matrix\n")
print(corr_df[cols].corr(method="spearman"))

Rows: 89405
Columns: ['seendate', 'url', 'title', 'language', 'domain', 'socialimage', 'company', 'ticker', 'date', 'sentiment_score', 'sentiment_confidence', 'sentiment_label', 'article_date', 'price_date', 'next_open', 'next_high', 'next_low', 'next_close', 'next_adj_close', 'next_volume']
Date range: 2024-02-09 00:00:00 -> 2026-02-24 00:00:00
Unique ticker-days: 1401

Pearson Correlation Matrix

                 sentiment_score  ret_cs_std  ret_cs_mad  ret_mean
sentiment_score         1.000000   -0.047106   -0.030416 -0.110070
ret_cs_std             -0.047106    1.000000    0.968739  0.284790
ret_cs_mad             -0.030416    0.968739    1.000000  0.342455
ret_mean               -0.110070    0.284790    0.342455  1.000000

Spearman Correlation Matrix

                 sentiment_score  ret_cs_std  ret_cs_mad  ret_mean
sentiment_score         1.000000   -0.011795    0.006417 -0.082935
ret_cs_std             -0.011795    1.000000    0.963408  0.146851
ret_cs_mad              0.006417

### Rolling Correlations:

In [2]:
corr_df = corr_df.sort_values("price_date")

windows = [5, 10, 30]

for w in windows:
    
    corr_df[f"roll_corr_sent_std_{w}d"] = (
        corr_df["sentiment_score"]
        .rolling(w)
        .corr(corr_df["ret_cs_std"])
    )

    corr_df[f"roll_corr_sent_mad_{w}d"] = (
        corr_df["sentiment_score"]
        .rolling(w)
        .corr(corr_df["ret_cs_mad"])
    )

    corr_df[f"roll_corr_sent_mean_{w}d"] = (
        corr_df["sentiment_score"]
        .rolling(w)
        .corr(corr_df["ret_mean"])
    )

print("\nRolling correlation preview:\n")
print(
    corr_df[
        [
            "price_date",
            "roll_corr_sent_std_5d",
            "roll_corr_sent_std_10d",
            "roll_corr_sent_std_30d",
        ]
    ].tail(10)
)

# Diagnostics for sparsity and rolling window coverage

print("\nNon-null rolling counts:")
print(corr_df.filter(like="roll_corr").count())

print("\nFinal daily rows after MIN_TICKERS filter:", len(corr_df))
print("\nFirst available dates:")
print(corr_df["price_date"].sort_values().head(10))


Rolling correlation preview:

                   price_date  roll_corr_sent_std_5d  roll_corr_sent_std_10d  \
113 2026-02-06 00:00:00+00:00              -0.586134               -0.677964   
114 2026-02-09 00:00:00+00:00              -0.501115               -0.752494   
115 2026-02-10 00:00:00+00:00              -0.776997               -0.773096   
116 2026-02-11 00:00:00+00:00              -0.844099               -0.776686   
117 2026-02-12 00:00:00+00:00              -0.632947               -0.704188   
118 2026-02-17 00:00:00+00:00              -0.651786               -0.738171   
119 2026-02-18 00:00:00+00:00               0.833297               -0.322969   
120 2026-02-19 00:00:00+00:00               0.243116               -0.487073   
121 2026-02-23 00:00:00+00:00               0.084616               -0.504096   
122 2026-02-24 00:00:00+00:00               0.031680               -0.159640   

     roll_corr_sent_std_30d  
113               -0.436394  
114               -0.423637 

### Export Summary Tables 

In [3]:
summary_cols = [
    "price_date",
    "sentiment_score",
    "ret_cs_std",
    "ret_cs_mad",
    "ret_mean",
    "n_tickers_returns",
    "coverage_ratio",
    "roll_corr_sent_std_5d",
    "roll_corr_sent_std_10d",
    "roll_corr_sent_std_30d",
    "roll_corr_sent_mad_5d",
    "roll_corr_sent_mad_10d",
    "roll_corr_sent_mad_30d",
    "roll_corr_sent_mean_5d",
    "roll_corr_sent_mean_10d",
    "roll_corr_sent_mean_30d",
]

summary_df = corr_df[summary_cols].copy()

print("\nRows exported:", len(summary_df))
print("Date range:", summary_df["price_date"].min(), "->", summary_df["price_date"].max())

output_path = "correlation_summary.csv"
summary_df.to_csv(output_path, index=False)

print(f"Saved summary table to {output_path}")
print(summary_df.head())


Rows exported: 123
Date range: 2024-02-09 00:00:00+00:00 -> 2026-02-24 00:00:00+00:00
Saved summary table to correlation_summary.csv
                 price_date  sentiment_score  ret_cs_std  ret_cs_mad  \
0 2024-02-09 00:00:00+00:00         0.014184    0.011921    0.009208   
1 2024-02-22 00:00:00+00:00        -0.071803    0.016797    0.011231   
2 2024-02-26 00:00:00+00:00        -0.034534    0.020499    0.011775   
3 2024-03-04 00:00:00+00:00        -0.077389    0.021471    0.014472   
4 2024-03-11 00:00:00+00:00        -0.043588    0.014154    0.010947   

   ret_mean  n_tickers_returns  coverage_ratio  roll_corr_sent_std_5d  \
0  0.011527                  7             1.0                    NaN   
1  0.012547                  7             1.0                    NaN   
2 -0.003908                  7             1.0                    NaN   
3 -0.009831                  7             1.0                    NaN   
4 -0.003249                  7             1.0              -0.68378